<a href="https://colab.research.google.com/github/m2nwo079/global-challenger-2026/blob/main/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/globalchallenger'
WORK = '/content/work'

for p in [f'{BASE}/notebooks', f'{BASE}/src', f'{BASE}/data/db',
          f'{BASE}/output', f'{BASE}/figures',
          f'{WORK}/data/raw', f'{WORK}/data/db']:
    os.makedirs(p, exist_ok=True)

print('BASE:', os.path.isdir(BASE))
print('WORK:', os.path.isdir(WORK))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: True
WORK: True


In [ ]:
!df -h /content | tail -1
!free -g | head -2
!sqlite3 --version

overlay         108G   22G   87G  20% /
               total        used        free      shared  buff/cache   available
Mem:              12           0           6           0           5          11
3.37.2 2022-01-06 13:25:41 872ba256cbf61d9290b571c0e6d82a20c224ca3ad82971edc46b29818d5dalt1


In [ ]:
%cd {WORK}/data/raw
!curl -L -o criteo.zip http://go.criteo.net/criteo-research-attribution-dataset.zip
!ls -lh

/content/work/data/raw
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  622M  100  622M    0     0  30.6M      0  0:00:20  0:00:20 --:--:-- 37.0M
total 1.3G
-rw-r--r-- 1 root root 623M Sep 13  2017 criteo_attribution_dataset.tsv.gz
-rw-r--r-- 1 root root 623M Aug  3 15:26 criteo.zip
-rw-r--r-- 1 root root  24K Oct  5  2017 Experiments.ipynb
-rw-r--r-- 1 root root 4.5K Oct  5  2017 README.md


In [ ]:
!unzip -o criteo.zip
!ls -lh
!cat README.md

Archive:  criteo.zip
  inflating: Experiments.ipynb       
  inflating: README.md               
  inflating: criteo_attribution_dataset.tsv.gz  
total 1.3G
-rw-r--r-- 1 root root 623M Sep 13  2017 criteo_attribution_dataset.tsv.gz
-rw-r--r-- 1 root root 623M Aug  3 15:26 criteo.zip
-rw-r--r-- 1 root root  24K Oct  5  2017 Experiments.ipynb
-rw-r--r-- 1 root root 4.5K Oct  5  2017 README.md
# Criteo Attribution Modeling for Bidding Dataset

This dataset is released along with the paper:

"Attribution Modeling Increases Efficiency of Bidding in Display Advertising"
**Eustache Diemert&ast;, Julien Meynet&ast; (Criteo Research), Damien Lefortier (Facebook), Pierre Galland (Criteo) &ast;authors contributed equally**

This work was published in: 2017 AdKDD & TargetAd Workshop, in conjunction with The 23rd ACM SIGKDD Conference on Knowledge Discovery and Data Mining (KDD 2017) (https://adkdd17.wixsite.com/adkddtargetad2017)

When using this dataset, please cite the paper with following bibte

In [ ]:
GZ = 'criteo_attribution_dataset.tsv.gz'

!zcat {GZ} | head -3

timestamp	uid	campaign	conversion	conversion_timestamp	conversion_id	attribution	click	click_pos	click_nb	cost	cpo	time_since_last_click	cat1	cat2	cat3	cat4	cat5	cat6	cat7	cat8	cat9
0	20073966	22589171	0	-1	-1	0	0	-1	-1	1e-05	0.390793596019	-1	5824233	9312274	3490278	29196072	11409686	1973606	25162884	29196072	29196072
2	24607497	884761	0	-1	-1	0	0	-1	-1	1e-05	0.0596001963727	423858	30763035	9312274	14584482	29196072	11409686	1973606	22644417	9312274	21091111


In [ ]:
HAS_HEADER = True
UID_TYPE   = 'INTEGER'

DB   = f'{WORK}/data/db/criteo.db'
SLIM = f'{WORK}/data/raw/attribution_slim.tsv'

print(HAS_HEADER, UID_TYPE)
print(DB)

True INTEGER
/content/work/data/db/criteo.db


In [ ]:
%cd {WORK}/data/raw
!ls -lh
!zcat criteo_attribution_dataset.tsv.gz | cut -f1-13 > attribution_slim.tsv
# !cut -f1-13 criteo_attribution_dataset.tsv > attribution_slim.tsv

!ls -lh attribution_slim.tsv
!head -2 attribution_slim.tsv
!wc -l attribution_slim.tsv


/content/work/data/raw
total 1.9G
-rw-r--r-- 1 root root 1.3G Aug  3 15:28 attribution_slim.tsv
-rw-r--r-- 1 root root 623M Sep 13  2017 criteo_attribution_dataset.tsv.gz
-rw-r--r-- 1 root root  24K Oct  5  2017 Experiments.ipynb
-rw-r--r-- 1 root root 4.5K Oct  5  2017 README.md
-rw-r--r-- 1 root root 1.3G Aug  3 15:34 attribution_slim.tsv
timestamp	uid	campaign	conversion	conversion_timestamp	conversion_id	attribution	click	click_pos	click_nb	cost	cpo	time_since_last_click
0	20073966	22589171	0	-1	-1	0	0	-1	-1	1e-05	0.390793596019	-1
16468028 attribution_slim.tsv


In [ ]:
import sqlite3

con = sqlite3.connect(DB)
con.executescript(f"""
PRAGMA journal_mode = OFF;
PRAGMA synchronous = OFF;
PRAGMA cache_size = -800000;
PRAGMA temp_store = MEMORY;

DROP TABLE IF EXISTS attribution;
CREATE TABLE attribution (
    timestamp             INTEGER,
    uid                   {UID_TYPE},
    campaign              INTEGER,
    conversion            INTEGER,
    conversion_timestamp  INTEGER,
    conversion_id         INTEGER,
    attribution           INTEGER,
    click                 INTEGER,
    click_pos             INTEGER,
    click_nb              INTEGER,
    cost                  REAL,
    cpo                   REAL,
    time_since_last_click REAL
);
""")
con.commit()
print(con.execute("PRAGMA table_info(attribution)").fetchall())

[(0, 'timestamp', 'INTEGER', 0, None, 0), (1, 'uid', 'INTEGER', 0, None, 0), (2, 'campaign', 'INTEGER', 0, None, 0), (3, 'conversion', 'INTEGER', 0, None, 0), (4, 'conversion_timestamp', 'INTEGER', 0, None, 0), (5, 'conversion_id', 'INTEGER', 0, None, 0), (6, 'attribution', 'INTEGER', 0, None, 0), (7, 'click', 'INTEGER', 0, None, 0), (8, 'click_pos', 'INTEGER', 0, None, 0), (9, 'click_nb', 'INTEGER', 0, None, 0), (10, 'cost', 'REAL', 0, None, 0), (11, 'cpo', 'REAL', 0, None, 0), (12, 'time_since_last_click', 'REAL', 0, None, 0)]


In [ ]:
%%time
import csv, itertools

cur = con.cursor()
sql = "INSERT INTO attribution VALUES (" + ",".join("?" * 13) + ")"
n = 0

with open(SLIM, newline='') as f:
    r = csv.reader(f, delimiter='\t')
    if HAS_HEADER:
        next(r)
    while True:
        chunk = list(itertools.islice(r, 200_000))
        if not chunk:
            break
        cur.executemany(sql, chunk)
        con.commit()
        n += len(chunk)
        print(f'{n:,}', end='\r')

con.commit()
print(f'\nLoaded {n:,} rows')

16,468,027
Loaded 16,468,027 rows
CPU times: user 2min 38s, sys: 9.38 s, total: 2min 48s
Wall time: 2min 53s


In [ ]:
import pandas as pd
print(pd.read_sql_query("""
SELECT typeof(timestamp) ts, typeof(uid) uid, typeof(click) clk,
       typeof(cost) cost, typeof(click_pos) cpos
FROM attribution LIMIT 1
""", con))

print(pd.read_sql_query(
    "SELECT MIN(timestamp) mn, MAX(timestamp) mx FROM attribution", con))

        ts      uid      clk  cost     cpos
0  integer  integer  integer  real  integer
   mn       mx
0   0  2671199


In [ ]:
import pandas as pd

print(pd.read_sql_query("""
SELECT typeof(timestamp) ts, typeof(uid) uid, typeof(campaign) camp,
       typeof(click) clk, typeof(cost) cost, typeof(cpo) cpo,
       typeof(time_since_last_click) tslc
FROM attribution LIMIT 1
""", con))

print(pd.read_sql_query(
    "SELECT MIN(timestamp) mn, MAX(timestamp) mx FROM attribution", con))

        ts      uid     camp      clk  cost   cpo  tslc
0  integer  integer  integer  integer  real  real  real
   mn       mx
0   0  2671199


In [ ]:
%%time
print(pd.read_sql_query("""
SELECT COUNT(*) impressions,
       COUNT(DISTINCT campaign) campaigns,
       COUNT(DISTINCT uid) users,
       AVG(click) ctr,
       COUNT(DISTINCT CASE WHEN conversion = 1 THEN conversion_id END) conversions
FROM attribution
""", con).T)

                        0
impressions  1.646803e+07
campaigns    6.750000e+02
users        6.142256e+06
ctr          3.611582e-01
conversions  4.358100e+05
CPU times: user 38.8 s, sys: 704 ms, total: 39.5 s
Wall time: 43.3 s


In [ ]:
print(pd.read_sql_query("""
SELECT COUNT(DISTINCT CASE WHEN attribution = 1 THEN conversion_id END) attributed,
       COUNT(DISTINCT CASE WHEN conversion  = 1 THEN conversion_id END) all_conv
FROM attribution
""", con).T)

print(pd.read_sql_query("SELECT click, COUNT(*) n FROM attribution GROUP BY click", con))

print(pd.read_sql_query("""
SELECT AVG(c) avg_touchpoints FROM (
  SELECT conversion_id, COUNT(*) c FROM attribution
  WHERE conversion_id >= 0 GROUP BY conversion_id)
""", con))

                 0
attributed  236331
all_conv    435810
   click         n
0      0  10520464
1      1   5947563
   avg_touchpoints
0          1.84988


In [ ]:
con.executescript("""
DROP TABLE IF EXISTS attribution_sample;
CREATE TABLE attribution_sample AS
SELECT * FROM attribution WHERE uid % 20 = 0;
""")
con.commit()

print(pd.read_sql_query(
    "SELECT COUNT(*) rows, COUNT(DISTINCT uid) users FROM attribution_sample", con))

     rows   users
0  822703  307095


In [ ]:
rules = """# Aggregation Rules

Definitions fixed before analysis, cross-checked against `Experiments.ipynb`
(the reproduction notebook shipped with the Criteo Attribution dataset).
Every query in this repository follows these rules.

## 1. Impression order

Impression order `k` is counted per `(uid, campaign)`, not per `uid`.
Counting per `uid` mixes impressions from different campaigns into one
sequence, which is not what "repeated exposure to the same ad" means.

## 2. Time to conversion

The gap is `conversion_timestamp - timestamp`, applied only to rows where
`conversion = 1`. There is no separate click-timestamp column in this
dataset; the impression timestamp is used. This matches the authors' code:

    df.loc[df.conversion == 1, 'gap_click_sale'] = \\
        df.conversion_timestamp - df.timestamp

Verified: zero rows with `conversion = 1` have
`conversion_timestamp < timestamp`.

## 3. Conversion counting

Conversions are deduplicated by `conversion_id`. A single conversion spans
multiple impression rows, so counting rows inflates the total. Note that
`conversion_id` is `-1` when no conversion occurred, so a bare
`COUNT(DISTINCT conversion_id)` counts `-1` as one conversion.

## 4. CTR denominator

The denominator is the number of impressions, not the number of users.

## 5. The `-1` sentinel

`-1` marks a missing value in `conversion_timestamp`, `conversion_id`,
`click_pos`, `click_nb`, and `time_since_last_click`. Always filter with
`>= 0` before computing means, minima, maxima, or time differences. The
authors apply the same mask:

    previous_tslc_mask = (df.time_since_last_click >= 0)

## 6. Day derivation

`day = floor(timestamp / 86400)`. Timestamps are second offsets from the
first impression, matching the authors' definition.

## Attribution rules (module B)

Taken verbatim from the authors' notebook rather than defined ad hoc:

    last_click  = attribution * (click_pos == click_nb - 1)
    first_click = attribution * (click_pos == 0)
    all_clicks  = attribution
    uniform     = attribution / click_nb

All four are conditioned on `attribution = 1`, so the population for
module B is 236,331 attributed conversions, not the 435,810 total
conversions.

## Measured baselines

| Metric | Value |
| --- | --- |
| Impressions | 16,468,027 |
| Users | 6,142,256 |
| Campaigns | 675 |
| Clicks | 5,947,563 (CTR 36.12%) |
| Distinct conversions (`conversion = 1`) | 435,810 |
| Distinct attributed conversions | 236,331 |
| Attributed share | 54.2% |
| Max timestamp | 2,671,199 (30.9 days) |

The dataset README states 45K conversions, which does not match the
distributed file under any counting definition tested. Impression count,
campaign count, and impressions-per-user (2.68) all match published
figures exactly, so the file is treated as intact and 435,810 is used.

CTR of 36.12% reflects the subsampling described in the dataset README.
Absolute click rates are not interpreted; only relative change across
impression order is used.
"""

with open(f'{BASE}/AGGREGATION_RULES.md', 'w') as f:
    f.write(rules)
print(rules)

# Aggregation Rules

Definitions fixed before analysis, cross-checked against `Experiments.ipynb`
(the reproduction notebook shipped with the Criteo Attribution dataset).
Every query in this repository follows these rules.

## 1. Impression order

Impression order `k` is counted per `(uid, campaign)`, not per `uid`.
Counting per `uid` mixes impressions from different campaigns into one
sequence, which is not what "repeated exposure to the same ad" means.

## 2. Time to conversion

The gap is `conversion_timestamp - timestamp`, applied only to rows where
`conversion = 1`. There is no separate click-timestamp column in this
dataset; the impression timestamp is used. This matches the authors' code:

    df.loc[df.conversion == 1, 'gap_click_sale'] = \
        df.conversion_timestamp - df.timestamp

Verified: zero rows with `conversion = 1` have
`conversion_timestamp < timestamp`.

## 3. Conversion counting

Conversions are deduplicated by `conversion_id`. A single conversion spans
multipl

In [ ]:
!cp {DB} /content/drive/MyDrive/globalchallenger/data/db/criteo.db
!ls -lh /content/drive/MyDrive/globalchallenger/data/db

total 890M
-rw------- 1 root root 890M Aug  3 16:17 criteo.db
